# Sparse Adam Optimizer

3DGS의 학습 속도를 **2.7배** 향상시키는 Sparse Adam 옵티마이저에 대해 알아봅니다.

## 배경: October 2024 Update

2024년 10월, 3DGS 공식 저장소에 Taming-3DGS (SIGGRAPH Asia 2024)의 최적화 기법이 통합되었습니다:

| 기능 | 설명 | 속도 향상 |
|------|------|-----------|
| **Fused SSIM** | CUDA 가속 SSIM 계산 | ×1.6 |
| **Sparse Adam** | Visibility 기반 sparse 업데이트 | ×2.7 |

---

## 핵심 아이디어

### 문제점: 기존 Adam의 비효율성

매 학습 iteration에서:
1. **하나의 카메라 뷰**만 렌더링
2. 전체 Gaussian 중 **일부만** 해당 뷰에서 보임
3. 보이지 않는 Gaussian은 **gradient = 0**
4. 그러나 기존 Adam은 **모든 Gaussian을 업데이트**

In [ ]:
# 기존 Adam: 매 iteration마다 "모든" Gaussian 업데이트
gaussians.optimizer.step()  # 수백만 개 Gaussian 전체 연산



### 해결책: Sparse Adam

**"보이지 않는 Gaussian은 업데이트하지 않는다"**

In [ ]:
# Sparse Adam: "보이는" Gaussian만 업데이트
visible = radii > 0  # 현재 카메라에서 보이는 Gaussian만
gaussians.optimizer.step(visible, radii.shape[0])  # 보이는 것만 업데이트



### 비교

| 구분 | 기존 Adam | Sparse Adam |
|------|-----------|-------------|
| 업데이트 대상 | **모든** Gaussian | **보이는** Gaussian만 |
| GPU 연산 | 불필요한 연산 포함 | 필요한 연산만 |
| 수학적 동등성 | - | O (gradient=0이면 업데이트 불필요) |
| 속도 | 기준 | **×2.7 빠름** |

---

## Visibility 판정

### 언제 Gaussian이 "보이는가"?

렌더링 전처리(`cuda_rasterizer/forward.cu`)에서 각 Gaussian은 여러 컬링 테스트를 거칩니다:

In [ ]:
__global__ void preprocessCUDA(..., int* radii, ...) {
    // 초기화
    radii[idx] = 0;

    // Frustum Culling
    if (!in_frustum(idx, orig_points, viewmatrix, projmatrix, ...))
        return;

    // Covariance Check
    if (det == 0.0f)
        return;

    // Tile Coverage
    // 화면을 16x16 픽셀 타일로 나누었을 때, Gaussian이 몇 개의 타일에 걸치는지 계산
    getRect(point_image, my_radius, rect_min, rect_max, grid);
    if ((rect_max.x - rect_min.x) * (rect_max.y - rect_min.y) == 0)
        return; // 0개 타일 → 화면 밖이거나 너무 작음 → invisible

    // Passed all checks
    radii[idx] = my_radius;
}



> **Note**: 이 단계의 Visibility는 **Geometric Visibility**(시야 내 존재 여부)만을 의미합니다. 다른 Gaussian에 가려져(Occlusion) 실제로는 보이지 않더라도, Frustum 안에 있고 타일과 겹친다면 `radii > 0`이 되어 업데이트 대상에 포함됩니다. 다만, 가려진 Gaussian은 Gradient가 0이므로 값은 변하지 않습니다.

### Visibility 흐름

Gaussian이 유효하려면 다음 조건을 **모두** 만족해야 합니다:

1. **Frustum Culling**: 카메라 시야각(Frustum) 내부에 중심점이 위치해야 함
2. **Covariance Validity**: 2D 공분산 행렬이 유효해야 함 (역행렬 존재, determinant != 0)
3. **Tile Touched > 0**: Screen space에서 Gaussian의 2D Bounding Box가 **16x16 픽셀 타일**을 하나라도 덮어야 함
   - `rect_min`, `rect_max`: Gaussian 영역의 타일 좌표 범위
   - `tiles_touched`: `(max.x - min.x) * (max.y - min.y)` (덮고 있는 타일의 개수)
   - 이 값이 0이면 렌더링 파이프라인에서 제외됨 (radii = 0)

---

## CUDA 구현

### Python → CUDA 연결

In [ ]:
# train.py
from diff_gaussian_rasterization import SparseGaussianAdam

# Sparse Adam 사용 조건 체크
use_sparse_adam = (opt.optimizer_type == "sparse_adam" and 
                   SPARSE_ADAM_AVAILABLE)

# 학습 루프에서
if use_sparse_adam:
    visible = radii > 0  # boolean mask
    gaussians.optimizer.step(visible, radii.shape[0])
else:
    gaussians.optimizer.step()  # 기존 방식



### SparseGaussianAdam 클래스

In [ ]:
# diff_gaussian_rasterization/__init__.py
class SparseGaussianAdam(torch.optim.Adam):
    def __init__(self, params, lr, eps):
        super().__init__(params=params, lr=lr, eps=eps)
    
    @torch.no_grad()
    def step(self, visibility, N):
        for group in self.param_groups:
            lr = group["lr"]
            eps = group["eps"]

            assert len(group["params"]) == 1, "more than one tensor in group"
            param = group["params"][0]
            if param.grad is None:
                continue

            # Lazy state initialization
            state = self.state[param]
            if len(state) == 0:
                state['step'] = torch.tensor(0.0, dtype=torch.float32)
                state['exp_avg'] = torch.zeros_like(param, memory_format=torch.preserve_format)
                state['exp_avg_sq'] = torch.zeros_like(param, memory_format=torch.preserve_format)

            stored_state = self.state.get(param, None)
            exp_avg = stored_state["exp_avg"]
            exp_avg_sq = stored_state["exp_avg_sq"]
            M = param.numel() // N
            _C.adamUpdate(param, param.grad, exp_avg, exp_avg_sq, 
                         visibility, lr, 0.9, 0.999, eps, N, M)



### CUDA 커널 (`adam.cu`)

In [ ]:
__global__ void adamUpdateCUDA(
    float* __restrict__ param,
    const float* __restrict__ param_grad,
    float* __restrict__ exp_avg,
    float* __restrict__ exp_avg_sq,
    const bool* tiles_touched,
    const float lr,
    const float b1,
    const float b2,
    const float eps,
    const uint32_t N,
    const uint32_t M) {

    auto p_idx = cg::this_grid().thread_rank();
    const uint32_t g_idx = p_idx / M;
    if (g_idx >= N) return;
    
    if (tiles_touched[g_idx]) {
        float Register_param_grad = param_grad[p_idx];
        float Register_exp_avg = exp_avg[p_idx];
        float Register_exp_avg_sq = exp_avg_sq[p_idx];
        
        // m_t = β₁ * m_{t-1} + (1 - β₁) * g_t
        Register_exp_avg = b1 * Register_exp_avg + (1.0f - b1) * Register_param_grad;
        
        // v_t = β₂ * v_{t-1} + (1 - β₂) * g_t²
        Register_exp_avg_sq = b2 * Register_exp_avg_sq + (1.0f - b2) * Register_param_grad * Register_param_grad;
        
        // θ_t = θ_{t-1} - lr * m_t / (√v_t + ε)
        float step = -lr * Register_exp_avg / (sqrt(Register_exp_avg_sq) + eps);

        param[p_idx] += step;
        exp_avg[p_idx] = Register_exp_avg;
        exp_avg_sq[p_idx] = Register_exp_avg_sq;
    }
}



---

## 사용 방법

### 1. 설치 - [주의] 3dgs_accel 브랜치로 전환

In [ ]:
# 3dgs_accel 브랜치로 전환
cd submodules/diff-gaussian-rasterization
git checkout 3dgs_accel
pip install .



### 2. 학습 실행

In [ ]:
# Sparse Adam 사용
python train.py -s <path_to_dataset> --optimizer_type sparse_adam

# 기존 방식 (default)
python train.py -s <path_to_dataset>



## 참고 자료

### Papers
- Mallick et al. "Taming 3DGS: High-Quality Radiance Fields with Limited Resources" (SIGGRAPH Asia 2024)
- Kerbl et al. "3D Gaussian Splatting for Real-Time Radiance Field Rendering" (2023)

### Code
- [Taming-3DGS GitHub](https://github.com/humansensinglab/taming-3dgs)
- [3DGS Official Repository](https://github.com/graphdeco-inria/gaussian-splatting)

---